# Gold Dimension: Trade (dim_trade)

- **Purpose**: Transforms `silver.trade` into the central `gold.dim_trade` hub. Resolves all dimensional surrogate keys (SKs) by executing point-in-time temporal joins against the historical dimension tables based on the trade execution timestamp (`T_DTS`).
- **Business Context**: PWG Pipeline - Trade Domain. Despite the `dim_` prefix, this functions as a degenerate dimension or core fact table storing actual trade events. It is the most complex table in the pipeline and acts as the central hub that subsequent tables (like `fact_holdings`) cascade keys from.
- **Execution Frequency**: Per Batch (Incremental CDC)
- **Inputs**: `silver.trade` (plus temporal lookups against all 5 Gold dimensions)
- **Outputs**: `gold.dim_trade` (MERGE pattern on natural key `TradeID`)
- **Dependencies**: **CRITICAL - MUST RUN LAST.** Depends on `dim_date`, `dim_time`, `dim_broker`, `dim_company`, `dim_security`, `dim_customer`, and `dim_account` being 100% complete.
- **Expected Row Count**: 1,302,248 (Validated via T_NEW B1 + B2 + B3)


> Imported our operations notebook which include all the functions and all

In [0]:
%run ../../02_common_utils/operations

In [0]:
# configuring widgets for ease of use and reusability
dbutils.widgets.text("env_catalog", "charles_schwab_retailbrokerage_dev_team_lemma")
dbutils.widgets.text("batch_id","1")

batch_id = dbutils.widgets.get("batch_id")
catalog = dbutils.widgets.get("env_catalog")


silver_tbl       = f"{catalog}.silver.trade"
gold_tbl         = f"{catalog}.gold.dim_trade"

silver_tradehist = f"{catalog}.silver.trade_history"
dim_account_tbl  = f"{catalog}.gold.dim_account"
dim_security_tbl = f"{catalog}.gold.dim_security"
dim_broker_tbl   = f"{catalog}.gold.dim_broker"

In [0]:
# Importing the required functions 
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
# Logging functions for initial load
l_df = spark.sql(f"SELECT * FROM {silver_tbl} ORDER BY _load_ts DESC LIMIT 1")
carried_run_id = str(l_df.select("_run_id").first()[0])

log_pipeline_message(spark, carried_run_id, 'INFO', 'gold_dim_trade', 'Starting processing for standalone gold dim_trade')
start_pipeline_run(spark, carried_run_id, batch_id)
log_domain_run_status(spark, carried_run_id, batch_id, 'TRADE', 'RUNNING')

In [0]:
# Reading the silver trade table
df_trade = spark.read.table(silver_tbl).filter(col("_batch") == batch_id).alias("t")
df_trade.cache()

# Storing source count from silver
source_count = df_trade.count()

> **Optimization Note:** We avoid 5 expensive physical joins by leveraging **Cascading Keys** (extracting the Customer key from `dim_account` and Company key from `dim_security`) and **Smart Keys** (deriving Date/Time keys directly from the timestamp), **_significantly reducing cluster compute overhead._**

In [0]:
# Reading the necessary gold tables from other domains for temporal join
dim_account  = spark.table(dim_account_tbl).alias("a")
dim_security = spark.table(dim_security_tbl).alias("s")
dim_broker   = spark.table(dim_broker_tbl).alias("b")

In [0]:
# Joining dim_account table with silver trade table using temporal join
df_joined = df_trade.join(dim_account,  (col("t.T_CA_ID") == col("a.AccountID")) &
                                        (col("t.T_DTS") >= col("a.EffectiveDate")) &
                                        (col("t.T_DTS") < col("a.EndDate")),"left")

In [0]:
# Joining dim_security table with silver trade table using temporal join technique
df_joined = df_joined.join(dim_security,(col("t.T_S_SYMB") == col("s.Symbol")) &
                                        (col("t.T_DTS") >= col("s.EffectiveDate")) &
                                        (col("t.T_DTS") < col("s.EndDate")),"left")

In [0]:
# Joining dim_broker table with silver trade table normally left join
df_joined = df_joined.join(dim_broker,col("t.T_EXEC_NAME") == col("b.BrokerID"),"left")

In [0]:
# Selecting required columns for final gold tables
df_mapped = df_joined.select(
    col("t.T_ID").cast("bigint").alias("TradeID"),
    col("b.SK_BrokerID"),
    date_format(col("t.T_DTS"), "yyyyMMdd").cast("bigint").alias("SK_CreateDateID"),
    date_format(col("t.T_DTS"), "HHmmss").cast("bigint").alias("SK_CreateTimeID"),
    lit(None).cast("bigint").alias("SK_CloseDateID"),
    lit(None).cast("bigint").alias("SK_CloseTimeID"),
    col("t.T_ST_ID").alias("Status"),
    col("t.T_TT_ID").alias("Type"),
    col("t.T_IS_CASH").alias("CashFlag"),
    col("t.T_QTY").alias("Quantity"),
    col("t.T_BID_PRICE").alias("BidPrice"),
    col("t.T_EXEC_NAME").alias("ExecutedBy"),
    col("t.T_TRADE_PRICE").alias("TradePrice"),
    col("t.T_CHRG").alias("Fee"),
    col("t.T_COMM").alias("Commission"),
    col("t.T_TAX").alias("Tax"),
    col("s.SK_SecurityID"),
    col("s.SK_CompanyID"),  
    col("a.SK_AccountID"),
    col("a.SK_CustomerID"),
    col("t._batch"),
    current_timestamp().alias("_load_ts")
)

In [0]:
from pyspark.sql.window import Window

# OLD CODE:
# df_mapped.createOrReplaceTempView("trade_updates")

# FIX: Deduplicate by TradeID before MERGE — temporal join on dim_account
# produces multiple matches when EffectiveDate/EndDate ranges overlap for the same AccountID.
w = Window.partitionBy("TradeID").orderBy(col("SK_CreateDateID").desc())
df_deduped = df_mapped.withColumn("_rn", row_number().over(w)).filter(col("_rn") == 1).drop("_rn")

df_deduped.createOrReplaceTempView("trade_updates")

In [0]:
(
df_mapped.limit(0).write
.format("delta")
.mode("ignore") # in this mode "ignore" means do nothing if the table already exists
.saveAsTable(gold_tbl)
)

final_merge = spark.sql(f"""
    MERGE INTO {gold_tbl} target
    USING trade_updates source
    ON target.TradeID = source.TradeID
    WHEN MATCHED THEN UPDATE SET *
    WHEN NOT MATCHED THEN INSERT *
""")

# Displaying mege metrics 
display(final_merge)

In [0]:
# Using trade history table to get the close date and time
spark.sql(f"""
    SELECT 
        CAST(TH_T_ID AS BIGINT) AS TradeID,
        CAST(date_format(TH_DTS, 'yyyyMMdd') AS BIGINT) AS SK_CloseDateID,
        CAST(date_format(TH_DTS, 'HHmmss') AS BIGINT) AS SK_CloseTimeID
    FROM {silver_tradehist}
    WHERE TH_ST_ID IN ('CMPT', 'CNCL')
""").createOrReplaceTempView("trade_closes")

In [0]:
spark.sql(f"""
    MERGE INTO {gold_tbl} as target
    USING trade_closes as source
    ON target.TradeID = source.TradeID
    WHEN MATCHED THEN UPDATE SET 
        target.SK_CloseDateID = source.SK_CloseDateID,
        target.SK_CloseTimeID = source.SK_CloseTimeID
""")

In [0]:
target_count = spark.table(gold_tbl).count()
null_count = spark.sql(f"SELECT COUNT(*) FROM {gold_tbl} WHERE SK_BrokerID IS NULL").first()[0]

log_dq_result(spark, carried_run_id, gold_tbl, "Null SK_BrokerID Check", null_count, source_count)
log_domain_run_status(spark, carried_run_id, batch_id, 'TRADE', 'COMPLETED')
end_pipeline_run(spark, carried_run_id, 'SUCCESS')
log_pipeline_message(spark, carried_run_id, 'INFO', 'gold_dim_trade', 'Successfully completed standalone gold dim_trade')
log_gold_recon(spark, carried_run_id, gold_tbl, expected_count=1302248, actual_count=target_count)

In [0]:
try:
    # Operations Logging
    

    # Extract the carry-forwarded _run_id from dataframe
    carried_run_id = str(df_joined.select("t._run_id").first()[0])

    log_pipeline_recon(
        spark=spark,
        run_id=carried_run_id,
        batch_id=batch_id,
        domain="TRADE",
        table_name="dim_trade",
        source_layer="silver",
        target_layer="gold",
        source_count=int(source_count),
        target_count=int(target_count)
    )

    log_audit_event(
        spark=spark,
        run_id=carried_run_id,
        batch=batch_id,
        layer="gold",
        table_name="dim_trade",
        operation="OVERWRITE",
        rows_affected=int(target_count)
    )

    print("Done GOLD")
except Exception as e:
    print(f"Error during operations logging: {e}")